# 🎬 ComfyUI Master Notebook (Anime Txt2Img + AnimateDiff + Wan 2.1)
### **Google Colab GPU T4 · 15 GB VRAM · Optimizado para Alta Velocidad y 0 OOM**
---
**Instrucciones de Uso:**
1. Ejecuta la **Celda 1** para instalar ComfyUI y todas las extensiones necesarias.
2. Ejecuta la **Celda 2** para descargar los Checkpoints y LoRAs.
3. Ejecuta la **Celda 3** para descargar los componentes de Video (AnimateDiff + Wan 2.1).
4. Ejecuta la **Celda 4** para iniciar ComfyUI y abrir la interfaz gráfica mediante el enlace público de **Pinggy / Cloudflare**.

In [ ]:
# 1️⃣ INSTALACIÓN BASE DE COMFYUI + CUSTOM NODES + GOOGLE DRIVE
import os, sys, subprocess

# 1. Actualizar e instalar dependencias del sistema
!apt -y update -qq
!apt -y install aria2 ffmpeg git -qq

# 2. Clonar ComfyUI Base si no existe
if not os.path.exists('/content/ComfyUI'):
    print('🚀 Clonando ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q

# 3. Instalación de Nodos Personalizados (Custom Nodes)
CUSTOM_NODES = '/content/ComfyUI/custom_nodes'
os.makedirs(CUSTOM_NODES, exist_ok=True)

nodes = {
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-WD14-Tagger': 'https://github.com/pythongosssss/ComfyUI-WD14-Tagger',
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved',
    'ComfyUI_FizzNodes': 'https://github.com/FizzleDorf/ComfyUI_FizzNodes',
    'ComfyUI-WanVideoWrapper': 'https://github.com/Kijai/ComfyUI-WanVideoWrapper',
    'ComfyUI_IPAdapter_plus': 'https://github.com/cubiq/ComfyUI_IPAdapter_plus',
    'ComfyUI_Style_Aligned': 'https://github.com/leeguandong/ComfyUI_Style_Aligned'
}

for name, url in nodes.items():
    path = os.path.join(CUSTOM_NODES, name)
    if not os.path.exists(path):
        print(f'📦 Instalando {name}...')
        !git clone {url} {path}
        req_file = os.path.join(path, 'requirements.txt')
        if os.path.exists(req_file):
            !pip install -r {req_file} -q

# 4. Conectar Google Drive para guardar renders automáticamente
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs('/content/drive/MyDrive/ComfyUI/output', exist_ok=True)
!rm -rf /content/ComfyUI/output
!ln -sf /content/drive/MyDrive/ComfyUI/output /content/ComfyUI/output

print('\n✅ Celda 1 Completada: ComfyUI, IPAdapter y StyleAligned instalados.')


In [ ]:
# 2️⃣ DESCARGAR MODELOS Y LoRAs (Google Colab - Versión IllustriousXL Nativa Completa)
import requests, os
from tqdm import tqdm

os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

CIVITAI_TOKEN = "1b82d9ca1d1220d1e15a6ea0d1aafb97"  # ← TOKEN DE CIVITAI

MODEL_DIR = "/content/ComfyUI/models/checkpoints"
LORA_DIR = "/content/ComfyUI/models/loras"
VAE_DIR = "/content/ComfyUI/models/vae"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

def download_civitai(version_id, dest_dir, filename, token):
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        mb = os.path.getsize(dest) // 1_048_576
        print(f'ℹ️ {filename} ({mb} MB) ya existe')
        return

    url = f'https://civitai.com/api/download/models/{version_id}'
    r = requests.get(url, params={'token': token}, headers=headers, stream=True)

    if r.status_code == 200:
        total_size = int(r.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB

        with open(dest, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
        ) as bar:
            for chunk in r.iter_content(chunk_size=block_size):
                size = f.write(chunk)
                bar.update(size)
    else:
        print(f'❌ {filename}: error {r.status_code}')

def download_civitai_by_model(model_id, dest_dir, filename, token):
    try:
        api_url = f"https://civitai.com/api/v1/models/{model_id}"
        resp = requests.get(api_url, headers=headers, timeout=10).json()
        version_id = resp['modelVersions'][0]['id']
        print(f"🔍 Model ID {model_id} → Version ID detectado: {version_id}")
        download_civitai(version_id, dest_dir, filename, token)
    except Exception as e:
        print(f"⚠️ Error al resolver Model ID {model_id}: {e}")

# ═══════════════════════════════════════════
# Checkpoints Base (Illustrious XL & SD 1.5)
# ═══════════════════════════════════════════
download_civitai(2883731, MODEL_DIR, 'waiIllustriousSDXL_v170.safetensors', CIVITAI_TOKEN)
download_civitai(2073605, MODEL_DIR, 'hardcoreHentai_sd1V13Baked.safetensors', CIVITAI_TOKEN)
download_civitai(1723898, MODEL_DIR, 'UnholyDesireMixSinisterAesthetic_V5_Illustrious.safetensors', CIVITAI_TOKEN)
download_civitai(119057, MODEL_DIR, 'meinamix_v11.safetensors', CIVITAI_TOKEN)  # Anime 2D AnimateDiff SD1.5

# VAE para Illustrious XL / SDXL
download_civitai(289115, VAE_DIR, 'sdxl_vae.safetensors', CIVITAI_TOKEN)

# ═══════════════════════════════════════════
# LoRAs de SOPORTE (compartidos por todos)
# ═══════════════════════════════════════════
download_civitai(1295238, LORA_DIR, 'doublepenetration_r1.safetensors', CIVITAI_TOKEN)
download_civitai(1938300, LORA_DIR, 'penetration_depth.safetensors', CIVITAI_TOKEN)
download_civitai(1831724, LORA_DIR, 'Penis Size Slider - Illustrious - V5_alpha1.0_rank4_noxattn_last.safetensors', CIVITAI_TOKEN)

# ═══════════════════════════════════════════
# PERSONAJES — Stella Sora (todos Illustrious)
# ═══════════════════════════════════════════
# ── Originales ──
download_civitai(2364713, LORA_DIR, 'Stella-Virigia-v1.safetensors', CIVITAI_TOKEN)          # Virigia (default+bunny)
download_civitai(2354793, LORA_DIR, 'Shia_Stella_Sora.safetensors', CIVITAI_TOKEN)            # Shia
download_civitai(2359712, LORA_DIR, 'Stella-Bernina-v1.safetensors', CIVITAI_TOKEN)          # Bernina (maid+bunny)
download_civitai(2361013, LORA_DIR, 'Reisen_ridge_-_Stella_Sora.safetensors', CIVITAI_TOKEN) # Reisen

# ── Tanda Dovellys ──
download_civitai(2181759, LORA_DIR, 'Amber_Stella_Sora.safetensors', CIVITAI_TOKEN)          # Amber
download_civitai(2343433, LORA_DIR, 'Portia_Stella_Sora.safetensors', CIVITAI_TOKEN)         # Portia
download_civitai(2447255, LORA_DIR, 'Freesia_Stella-10.safetensors', CIVITAI_TOKEN)          # Freesia
download_civitai(2438645, LORA_DIR, 'Laru_Dovellys.safetensors', CIVITAI_TOKEN)              # Laru
download_civitai(2483282, LORA_DIR, 'Nazuka_Dovellys.safetensors', CIVITAI_TOKEN)            # Nazuka
download_civitai(2462796, LORA_DIR, 'Kaede_Dovellys.safetensors', CIVITAI_TOKEN)             # Kaede
download_civitai(2438241, LORA_DIR, 'Tilia_Dovellys.safetensors', CIVITAI_TOKEN)             # Tilia
download_civitai(2438577, LORA_DIR, 'Canace_Dovellys.safetensors', CIVITAI_TOKEN)            # Canace
download_civitai(2438730, LORA_DIR, 'Caramel_Dovellys.safetensors', CIVITAI_TOKEN)           # Caramel
download_civitai(2435223, LORA_DIR, 'Cosette_Dovellys.safetensors', CIVITAI_TOKEN)           # Cosette
download_civitai(2439838, LORA_DIR, 'FuyukaSS-10.safetensors', CIVITAI_TOKEN)                # Fuyuka
download_civitai(2967914, LORA_DIR, 'FFSS-10.safetensors', CIVITAI_TOKEN)                    # Firefly
download_civitai(2427827, LORA_DIR, 'IrisStellaSora_IXL.safetensors', CIVITAI_TOKEN)         # Iris
download_civitai(2427803, LORA_DIR, 'MistiqueStellaSora_IXL.safetensors', CIVITAI_TOKEN)     # Mistique

# ── Tanda MD (guía) ──
download_civitai(2427816, LORA_DIR, 'NazunaStellaSora_IXL.safetensors', CIVITAI_TOKEN)       # Nazuna
download_civitai(2341691, LORA_DIR, 'bastelina_stellasora-v01.safetensors', CIVITAI_TOKEN)   # Bastelina
download_civitai(2351932, LORA_DIR, 'Flora_Stella_Sora.safetensors', CIVITAI_TOKEN)          # Flora
download_civitai(2388063, LORA_DIR, 'tyrant_v2.safetensors', CIVITAI_TOKEN)                  # Tyrant (peso 0.8)
download_civitai(2963012, LORA_DIR, 'Otoha_stella_sora.safetensors', CIVITAI_TOKEN)          # Otoha

# ── Tanda MD v2 (Illustrious) ──
download_civitai(2854565, LORA_DIR, 'Chitose_Dovellys.safetensors', CIVITAI_TOKEN)           # Chitose (3 outfits)
download_civitai(2388751, LORA_DIR, 'Noya_stella_sora.safetensors', CIVITAI_TOKEN)           # Noya (3 outfits)

# LoRA Personaje Masculino Feo/Calvo NATIVO ILLUSTRIOUS (Model ID: 1259610)
download_civitai_by_model(1259610, LORA_DIR, 'faceless-ugly-man-illustriousxl-lora-nochekaiser.safetensors', CIVITAI_TOKEN)

# Slider de Oscuridad (-3.5) y Estabilizadores
download_civitai(1444863, LORA_DIR, 'darkness_slider.safetensors', CIVITAI_TOKEN)
download_civitai(2073647, LORA_DIR, 'stabilizer_animaginexl.safetensors', CIVITAI_TOKEN)
# Double Anal (Illustrious v2133303)
download_civitai(2133303, LORA_DIR, 'double_anal_ilxl_goofy.safetensors', CIVITAI_TOKEN)

# Double/Triple Vaginal (Illustrious v1139591)
download_civitai(1139591, LORA_DIR, 'concept_double_vaginal-ill_d.safetensors', CIVITAI_TOKEN)
# Poses y Efectos
download_civitai(1833287, LORA_DIR, 'amazon_position.safetensors', CIVITAI_TOKEN)
download_civitai(2167964, LORA_DIR, 'hand_holding_sex_pose.safetensors', CIVITAI_TOKEN)
download_civitai(1412807, LORA_DIR, 'cowgirl_position.safetensors', CIVITAI_TOKEN)
download_civitai(2186584, LORA_DIR, 'masturbation_position.safetensors', CIVITAI_TOKEN)
download_civitai(1291250, LORA_DIR, 'danglinglegs.safetensors', CIVITAI_TOKEN)
download_civitai(3008158, LORA_DIR, 'hentai_comic_generator.safetensors', CIVITAI_TOKEN)
download_civitai(1486887, LORA_DIR, 'add-detail-xl.safetensors', CIVITAI_TOKEN)
download_civitai(2769575, LORA_DIR, 'Face_Detailed_v2.safetensors', CIVITAI_TOKEN)
download_civitai(1192192, LORA_DIR, 'Ohogao_illustrious_v1.safetensors', CIVITAI_TOKEN)
download_civitai(1291450, LORA_DIR, 'excessivecum.safetensors', CIVITAI_TOKEN)
download_civitai(2369236, LORA_DIR, 'uterus_1.0.safetensors', CIVITAI_TOKEN)
download_civitai(1307519, LORA_DIR, 'xray (1).safetensors', CIVITAI_TOKEN)
download_civitai(2517707, LORA_DIR, 'cumtrail_pullingout_v4_for_ILXL.safetensors', CIVITAI_TOKEN)
download_civitai(1984258, LORA_DIR, 'lactationAlpha_2_Illustrious_DIM-16_sv_fro_0.95.safetensors', CIVITAI_TOKEN)

print(f'\n✅ Finalizado: Todos los modelos, VAE y LoRAs de IllustriousXL han sido descargados.')


In [ ]:
# 3️⃣ DESCARGA DE COMPONENTES DE VIDEO E IP-ADAPTER
import os

AD_MODELS = '/content/ComfyUI/models/animatediff_models'
AD_EVOLVED_MODELS = '/content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models'
DIFFUSION_DIR = '/content/ComfyUI/models/diffusion_models'
VAE_DIR = '/content/ComfyUI/models/vae'
CLIP_DIR = '/content/ComfyUI/models/clip'
CLIP_VISION_DIR = '/content/ComfyUI/models/clip_vision'
IPADAPTER_DIR = '/content/ComfyUI/models/ipadapter'
LORA_DIR = '/content/ComfyUI/models/loras'

for d in [AD_MODELS, AD_EVOLVED_MODELS, DIFFUSION_DIR, VAE_DIR, CLIP_DIR, CLIP_VISION_DIR, IPADAPTER_DIR, LORA_DIR]:
    os.makedirs(d, exist_ok=True)

# A. Modelos IP-Adapter Plus SDXL + CLIP Vision (ViT-H)
print('🎨 Descargando CLIP-Vision Encoder (ViT-H)...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors -d {CLIP_VISION_DIR} -o CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

print('🎨 Descargando IP-Adapter Plus SDXL (ViT-H)...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors -d {IPADAPTER_DIR} -o ip-adapter-plus_sdxl_vit-h.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter_sdxl_vit-h.safetensors -d {IPADAPTER_DIR} -o ip-adapter_sdxl_vit-h.safetensors

# B. Modelos de Movimiento AnimateDiff (SD 1.5 + SDXL/Illustrious)
print('🎬 Descargando AnimateDiff Motion Model SD 1.5 (mm_sd_v15_v2.ckpt)...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt -d {AD_MODELS} -o mm_sd_v15_v2.ckpt
!cp -f {AD_MODELS}/mm_sd_v15_v2.ckpt {AD_EVOLVED_MODELS}/mm_sd_v15_v2.ckpt 2>/dev/null || true

print('🎬 Descargando AnimateDiff Motion Model SDXL/Illustrious (mm_sdxl_v10_beta.ckpt)...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/guoyww/animatediff/resolve/main/mm_sdxl_v10_beta.ckpt -d {AD_MODELS} -o mm_sdxl_v10_beta.ckpt
!cp -f {AD_MODELS}/mm_sdxl_v10_beta.ckpt {AD_EVOLVED_MODELS}/mm_sdxl_v10_beta.ckpt 2>/dev/null || true

# C. Componentes Wan 2.1
print('🚀 Descargando Wan 2.1 Diffusion Model 1.3B FP8...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-T2V-1_3B_fp8_e4m3fn.safetensors -d {DIFFUSION_DIR} -o wan2.1_t2v_1.3B_fp8.safetensors
!ln -sf /content/ComfyUI/models/diffusion_models/wan2.1_t2v_1.3B_fp8.safetensors /content/ComfyUI/models/checkpoints/wan2.1_t2v_1.3B_fp8.safetensors

print('🚀 Descargando VAE Wan 2.1...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d {VAE_DIR} -o wan_2.1_vae.safetensors

print('🚀 Descargando UMT5 Text Encoder...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5-xxl-enc-fp8_e4m3fn.safetensors -d {CLIP_DIR} -o umt5-xxl-enc-fp8_e4m3fn.safetensors
!cp -f {CLIP_DIR}/umt5-xxl-enc-fp8_e4m3fn.safetensors {CLIP_DIR}/umt5_xxl_fp8.safetensors 2>/dev/null || true

print('🚀 Descargando Lightning LoRA 4-Step...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-T2V_FastWan_1_3B_bf16.safetensors -d {LORA_DIR} -o wan2.1_lightx2v_4step.safetensors

print('\n✅ Celda 3 Completada: AnimateDiff, IP-Adapter Plus SDXL y Wan 2.1 listos.')


In [ ]:
# 4️⃣ INICIAR COMFYUI + ENLACE PÚBLICO (PINGGY / CLOUDFLARE)
import threading, time, subprocess

%cd /content/ComfyUI

# Iniciar servidor de ComfyUI en segundo plano
def run_comfyui():
    !python main.py --listen 0.0.0.0 --port 8188 --enable-manager

t = threading.Thread(target=run_comfyui)
t.start()
time.sleep(5)

# Iniciar túnel de Cloudflare para obtener enlace público de acceso
print('\n🌐 Generando enlace público de acceso a la interfaz gráfica...')
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8188
